In [1]:
# Install Quantum, Neuromorphic, and AI libraries
!pip install -q cirq qsimcirq brian2 google-genai cuda-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 430.5/430.5 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.3/572.3 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 56.2 MB/s eta 0:00:00


In [2]:
import google.generativeai as genai
from google.colab import userdata
import cuda
# Configure Gemini
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
model = genai.GenerativeModel('gemini-2.5-flash')
def get_satellite_params(tle_data):
    prompt = f"Analyze this Starlink TLE data: {tle_data}. Extract the mean motion and inclination. Suggest a neuron firing threshold (0.5-1.0) and a quantum rotation angle (0-3.14) based on orbital stability."
    response = model.generate_content(prompt)
    return response.text

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
import google.generativeai as genai
from google.colab import userdata
import torch # Standard for CUDA checking in Colab

# 1. Securely load your API Key
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))

# 2. Initialize the Gemini 1.5 Flash Model
# This model is optimized for speed and cost-efficiency
model = genai.GenerativeModel('gemini-2.5-flash')

# 3. CUDA Health Check (for high-speed neuromorphic/quantum simulation)
cuda_available = torch.cuda.is_available()
print(f"CUDA Acceleration: {'[ACTIVE]' if cuda_available else '[OFFLINE]'}")
if cuda_available:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

CUDA Acceleration: [ACTIVE]
GPU Device: Tesla T4


In [4]:
from brian2 import *

def run_neural_simulation(threshold_val):
    start_scope()
    tau = 10*ms
    v_threshold = threshold_val * volt  # Parametrisized by Gemini

    # Fixed: Ensure '1' has consistent units with 'v'
    eqs = '''
    dv/dt = (1*volt-v)/tau : volt (unless refractory)
    '''

    G = NeuronGroup(1, eqs, threshold='v > v_threshold', reset='v = 0*volt', method='exact')
    M = StateMonitor(G, 'v', record=True)

    run(50*ms)
    return M.v[0] # Return voltage trace

In [5]:
import cirq
import qsimcirq

def run_quantum_circuit(angle):
    qubit = cirq.LineQubit(0)
    # Create a parameterized circuit
    circuit = cirq.Circuit(
        cirq.ry(angle)(qubit),  # Rotation dictated by Gemini
        cirq.measure(qubit, key='m')
    )

    # Use qsim for high-performance simulation
    simulator = qsimcirq.QSimSimulator()
    result = simulator.run(circuit, repetitions=100)
    return result.histogram(key='m')

In [6]:
import google.generativeai as genai
from google.colab import userdata

# Initialize API_KEY at the module level
API_KEY = None

# Fetch API key once at the module level to avoid TimeoutException
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel('gemini-2.0-flash')
except Exception as e:
    print(f"Error fetching GEMINI_API_KEY: {e}. Please ensure it's set in Colab secrets.")

# Global variable to store the selected model name to avoid re-discovering every time
_selected_gemini_model = None

def get_satellite_params(tle_data):
    global _selected_gemini_model

    if API_KEY is None:
        raise RuntimeError("GEMINI_API_KEY is not available.")

    # genai.configure(api_key=API_KEY) # No need to re-configure if already done globally

    if _selected_gemini_model is None:
        # Discover available models only once
        available_models = [m.name for m in genai.list_models()
                            if 'generateContent' in m.supported_generation_methods]
        if available_models:
            _selected_gemini_model = available_models[0] # Pick the first available model
            print(f"Dynamically selected Gemini model: {_selected_gemini_model}")
        else:
            raise RuntimeError("No Gemini models found that support generateContent.")

    # Use the dynamically selected model
    local_model = genai.GenerativeModel(_selected_gemini_model)

    # Precise instruction for Gemini reasoning
    prompt = f"""
    SYSTEM: You are a Quantum-Neuromorphic Analyst.
    DATA: {tle_data}

    TASK:
    1. Parse the TLE for Mean Motion and Inclination.
    2. Assess orbital stability.
    3. Output a precise JSON structure with:
       - 'neuron_threshold': (float 0.5-1.0)
       - 'quantum_angle': (float 0.0-3.14)
       - 'reasoning': (short explanation)
    """

    response = local_model.generate_content(prompt) # Use the local_model
    return response.text

# Example Execution
starlink_tle = "1 44713U 19074A   23310.518..."
analysis = get_satellite_params(starlink_tle)
print(analysis)

Dynamically selected Gemini model: models/gemini-2.5-flash
As a Quantum-Neuromorphic Analyst, I must first process the provided data.

**Analysis:**

1.  **Parse TLE for Mean Motion and Inclination:**
    The provided data `1 44713U 19074A 23310.518...` constitutes only the first line of a Two-Line Element (TLE) set. Mean Motion is found as the last field on TLE Line 2, and Inclination is the first field after the line number on TLE Line 2.
    **Conclusion:** Mean Motion and Inclination cannot be parsed from the single line of TLE data provided.

2.  **Assess orbital stability:**
    Without the Mean Motion (which indicates orbital period and thus approximate altitude) and Inclination, a precise assessment of orbital stability is impossible. These parameters are crucial for understanding atmospheric drag, gravitational perturbations, and potential resonance conditions.

    However, I can note the following from Line 1:
    *   **Satellite Catalog Number:** 44713
    *   **Internation

In [7]:
# 1. Sample Starlink TLE (Mock data for demonstration)
starlink_tle = "1 44713U 19074A   23310.518... 2 44713  53.0543 154.34..."

# 2. Get AI Insights
print("Analyzing with Gemini...")
insights = get_satellite_params(starlink_tle)
# Let's assume Gemini suggests: threshold=0.8, angle=1.57 (π/2)

# 3. Process with Brain2
print("Running Neuromorphic Simulation...")
voltages = run_neural_simulation(0.8)

# 4. Execute Quantum Optimization
print("Executing Quantum Circuit...")
q_results = run_quantum_circuit(1.57)

print("--- Engine Output ---")
print(f"Neuromorphic Signal Peak: {max(voltages)}")
print(f"Quantum State Distribution: {q_results}")

Analyzing with Gemini...


WARNING    Model equations use the "unless refractory" flag but no refractory keyword was given. [brian2.groups.neurongroup.no_refractory]


Running Neuromorphic Simulation...
Executing Quantum Circuit...
--- Engine Output ---
Neuromorphic Signal Peak: 0.79810348 V
Quantum State Distribution: Counter({1: 51, 0: 49})


In [8]:
!pip install -q cirq qsimcirq brian2 skyfield

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.4/370.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.7/235.7 kB 25.5 MB/s eta 0:00:00


In [9]:
import skyfield.api as sf # Use an alias for robustness

def get_starlink_telemetry():
    stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
    try:
        # Correctly create a loader instance using sf.build_downloader
        loader = sf.build_downloader(directory='.')
        satellites = loader.tle_file(stations_url)
        # Let's take the first active Starlink satellite found
        sat = satellites[0]

        ts = sf.load.timescale() # Use sf.load.timescale()
        t = ts.now()

        geocentric = sat.at(t)
        subpoint = geocentric.subpoint()

        return {
            "name": sat.name,
            "lat": subpoint.latitude.degrees,
            "lon": subpoint.longitude.degrees,
            "epoch": sat.epoch.utc_jpl()
        }
    except Exception as e:
        print(f"Connection error or TLE parsing error: {e}. Switching to fallback data.")
        return {"name": "STARLINK-DEBUG", "lat": 34.05, "lon": -118.24, "epoch": "N/A"}

In [10]:
from brian2 import *

def neural_signal_processor(raw_input_strength):
    start_scope()
    # Parameters for Leaky Integrate-and-Fire (LIF)
    tau = 10*ms
    v_rest = 0*mV
    v_threshold = 15*mV

    # Input is modeled as a Poisson stream based on Starlink signal strength
    input_group = PoissonGroup(10, rates=raw_input_strength*Hz)

    eqs = 'dv/dt = (v_rest - v) / tau : volt'
    processing_layer = NeuronGroup(1, eqs, threshold='v > v_threshold', reset='v = v_rest', method='exact')

    # Synaptic connection
    S = Synapses(input_group, processing_layer, on_pre='v += 2*mV')
    S.connect()

    # Monitor the spikes
    spike_mon = SpikeMonitor(processing_layer)
    run(100*ms)

    return spike_mon.count[0] # Return the spike count as a feature

In [11]:
import google.generativeai as genai

def gemini_bridge(telemetry, spike_count):
    prompt = f"""
    Telemetry: {telemetry}
    Neural Processor Spikes: {spike_count}

    Based on this satellite data, calculate a quantum phase shift (rotation angle in radians).
    Provide ONLY the numerical value between 0 and 3.14.
    """
    response = model.generate_content(prompt)
    return float(response.text.strip())

In [12]:
import re
import google.generativeai as genai
from google.colab import userdata
from skyfield.api import load # Removed build_downloader from import

# Configure Gemini API and model globally
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

# Dynamic model selection for robustness
_gemini_model_instance = None # Use a global-like variable for the model instance

def get_gemini_model():
    global _gemini_model_instance
    if _gemini_model_instance is None:
        available_models = [m.name for m in genai.list_models()
                            if 'generateContent' in m.supported_generation_methods]
        if available_models:
            # Prioritize 'flash' models if available for speed, otherwise take the first
            flash_models = [m for m in available_models if 'flash' in m.lower()]
            selected_model_name = flash_models[0] if flash_models else available_models[0]
            print(f"Dynamically selected Gemini model: {selected_model_name}")
            _gemini_model_instance = genai.GenerativeModel(selected_model_name)
        else:
            raise RuntimeError("No Gemini models found that support generateContent.")
    return _gemini_model_instance

def get_starlink_telemetry():
    stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
    try:
        # Simplified: Use load.tle_file directly, Skyfield handles downloading/caching
        satellites = load.tle_file(stations_url)
        # Select an active Starlink satellite
        sat = satellites[0]

        ts = load.timescale()
        t = ts.now()

        geocentric = sat.at(t)
        subpoint = geocentric.subpoint()

        return {
            "name": sat.name,
            "lat": subpoint.latitude.degrees,
            "lon": subpoint.longitude.degrees,
            "raw_tle": "N/A\nN/A" # Placeholder as raw_tle isn't used downstream
        }
    except Exception as e:
        print(f"Connection error or TLE parsing error: {e}. Switching to fallback data.")
        return {"name": "STARLINK-DEBUG", "lat": 34.05, "lon": -118.24, "raw_tle": "N/A\nN/A"}

def gemini_bridge(telemetry_dict, spike_count):
    # Get the dynamically selected model
    model_instance = get_gemini_model()

    prompt = f"""
    Satellite {telemetry_dict['name']} is at Lat: {telemetry_dict['lat']}, Lon: {telemetry_dict['lon']}.
    The Neuromorphic processor recorded {spike_count} spikes.

    Act as the Analytical Engine Controller. Calculate a quantum rotation angle (theta)
    between 0 and 3.14 to optimize signal phase.
    Return ONLY the number.
    """
    try:
        response = model_instance.generate_content(prompt) # Use the instance
        text_output = response.text.strip()

        # Use regex to find the first floating point number in case Gemini adds prose
        match = re.search(r"[-+]?\d*\.\d+|\d+", text_output)
        return float(match.group()) if match else 1.57 # Default to pi/2 if fails
    except Exception as e:
        print(f"Gemini bridge error: {e}. Returning default angle.")
        return 1.57 # Default to Pi/2 if any error occurs

# 1. Fetch live data
telemetry = get_starlink_telemetry()

# 2. Process through Neural Network (simulating a 50Hz signal input)
# neural_signal_processor is assumed to be defined elsewhere and works correctly
spike_feature = neural_signal_processor(50)

# 3. Gemini determines the Quantum Rotation
angle = gemini_bridge(telemetry, spike_feature)

# 4. Execute your Quantum Circuit using qsim
# run_quantum_circuit is assumed to be defined elsewhere and works correctly
quantum_output = run_quantum_circuit(angle)

print(f"--- Analytical Engine Results ---")
print(f"Satellite: {telemetry['name']}") # Access name from dict
print(f"Neural Spike Activity: {spike_feature}")
print(f"Quantum Phase Angle: {angle}")
print(f"Quantum Probability Distribution: {quantum_output}")

[#################################] 100% gp.php


Dynamically selected Gemini model: models/gemini-2.5-flash
--- Analytical Engine Results ---
Satellite: STARLINK-1008
Neural Spike Activity: 3
Quantum Phase Angle: 0.3487530927016598
Quantum Probability Distribution: Counter({0: 98, 1: 2})


In [13]:
from skyfield.api import load

def get_starlink_telemetry():
    # Create a loader instance to handle file downloads
    loader = load.build_downloader(directory='.')
    stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'

    # Use the loader instance, not the module function
    satellites = loader.tle_file(stations_url)

    # Filter for an active satellite (e.g., STARLINK-31)
    sat = satellites[0]
    ts = load.timescale()
    t = ts.now()

    # Get current position for the "Analytical Engine" input
    geocentric = sat.at(t)
    subpoint = geocentric.subpoint()

    return {
        "name": sat.name,
        "lat": subpoint.latitude.degrees,
        "lon": subpoint.longitude.degrees,
        "raw_tle": f"{sat.model.line1}\n{sat.model.line2}"
    }

In [14]:
import re

def gemini_bridge(telemetry_dict, spike_count):
    prompt = f"""
    Satellite {telemetry_dict['name']} is at Lat: {telemetry_dict['lat']}, Lon: {telemetry_dict['lon']}.
    The Neuromorphic processor recorded {spike_count} spikes.

    Act as the Analytical Engine Controller. Calculate a quantum rotation angle (theta)
    between 0 and 3.14 to optimize signal phase.
    Return ONLY the number.
    """
    response = model.generate_content(prompt)
    text_output = response.text.strip()

    # Use regex to find the first floating point number in case Gemini adds prose
    match = re.search(r"[-+]?\d*\.\d+|\d+", text_output)
    return float(match.group()) if match else 1.57 # Default to pi/2 if fails

In [15]:
import google.generativeai as genai
from google.colab import userdata
import re # for gemini_bridge regex
from skyfield.api import load # for get_starlink_telemetry
# Assuming brian2 and cirq are already imported or will be imported at top level

# --- Functions from other cells, consolidated for self-containation ---

# from o4T1GiiN76vB, modified for robustness
def get_starlink_telemetry():
    stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
    try:
        satellites = load.tle_file(stations_url)

        sat = None
        if isinstance(satellites, dict) and satellites: # Assume dictionary as per Skyfield docs
            sat = next(iter(satellites.values())) # Robustly get the first satellite object
        elif isinstance(satellites, list) and satellites: # Handle unexpected list case (from previous error)
            sat = satellites[0]

        if sat is None:
            raise ValueError("No satellite data found or data is empty.")

        ts = load.timescale()
        t = ts.now()
        geocentric = sat.at(t)
        subpoint = geocentric.subpoint()
        raw_tle_line1 = getattr(sat.model, 'line1', 'N/A')
        raw_tle_line2 = getattr(sat.model, 'line2', 'N/A')
        return {
            "name": sat.name,
            "lat": subpoint.latitude.degrees,
            "lon": subpoint.longitude.degrees,
            "raw_tle": f"{raw_tle_line1}\\n{raw_tle_line2}"
        }
    except Exception as e:
        print(f"Connection error or TLE parsing error: {e}. Switching to fallback data.")
        return {"name": "STARLINK-DEBUG", "lat": 34.05, "lon": -118.24, "raw_tle": "N/A\\nN/A"}

# From YLDhUr_X7toP - Gemini model setup, modified for robustness
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
_gemini_model_instance = None # Global for dynamic model selection

def get_gemini_model():
    global _gemini_model_instance
    if _gemini_model_instance is None:
        available_models = [m.name for m in genai.list_models()
                            if 'generateContent' in m.supported_generation_methods]

        models_to_try = []
        # Prioritize flash models
        flash_models = [m for m in available_models if 'flash' in m.lower()]
        if flash_models: models_to_try.extend(flash_models)
        # Add other available models as fallback
        non_flash_models = [m for m in available_models if 'flash' not in m.lower()]
        if non_flash_models: models_to_try.extend(non_flash_models)

        if not models_to_try:
            raise RuntimeError("No Gemini models found that support generateContent.")

        for model_name in models_to_try:
            try:
                # Test if the model can be instantiated and used
                temp_model = genai.GenerativeModel(model_name)
                # A small test to ensure it's truly usable and not just listed
                test_response = temp_model.generate_content("test", generation_config=genai.GenerationConfig(max_output_tokens=1))
                if test_response and test_response.candidates:
                    _gemini_model_instance = temp_model
                    print(f"Dynamically selected working Gemini model: {model_name}")
                    return _gemini_model_instance
            except Exception as e:
                print(f"Warning: Model {model_name} failed during test: {e}")
                continue # Try next model

        raise RuntimeError("Could not find any working Gemini model after testing.")
    return _gemini_model_instance

# From YLDhUr_X7toP - Gemini bridge function
def gemini_bridge(telemetry_dict, spike_count):
    model_instance = get_gemini_model() # Get the dynamically selected instance
    prompt = f"""
    Satellite {telemetry_dict['name']} is at Lat: {telemetry_dict['lat']}, Lon: {telemetry_dict['lon']}.
    The Neuromorphic processor recorded {spike_count} spikes.
    Act as the Analytical Engine Controller. Calculate a quantum rotation angle (theta)
    between 0 and 3.14 to optimize signal phase.
    Return ONLY the number.
    """
    try:
        response = model_instance.generate_content(prompt)
        text_output = response.text.strip()
        match = re.search(r"[-+]?\d*\.\d+|\d+", text_output)
        return float(match.group()) if match else 1.57 # Default to pi/2 if fails
    except Exception as e:
        print(f"Gemini bridge error: {e}. Returning default angle.")
        return 1.57 # Default to Pi/2

# --- Original execution logic for LcUhihyn75MN ---

# 1. Fetch live data (Corrected)
telemetry_data = get_starlink_telemetry()
print(f"Tracking: {telemetry_data['name']}")

# 2. Process through Brian2 (Neuromorphic Layer)
# We use the Latitude as a seed for 'signal noise'
# neural_signal_processor is assumed to be defined elsewhere and works correctly
spike_feature = neural_signal_processor(abs(telemetry_data['lat']) + 10)

# 3. Gemini Bridge (Orchestration Layer)
angle = gemini_bridge(telemetry_data, spike_feature)

# 4. Quantum Layer (Cirq + qsimcirq)
# run_quantum_circuit is assumed to be defined elsewhere and works correctly
quantum_output = run_quantum_circuit(angle)

print(f"--- Engine Output ---")
print(f"Position: {telemetry_data['lat']:.2f}, {telemetry_data['lon']:.2f}")
print(f"Neural Spikes: {spike_feature}")
print(f"Quantum Result: {quantum_output}")

Tracking: STARLINK-1008
Dynamically selected working Gemini model: models/gemini-2.5-flash
--- Engine Output ---
Position: 50.07, 175.49
Neural Spikes: 1
Quantum Result: Counter({1: 55, 0: 45})


In [16]:
from skyfield.api import load # Removed build_downloader from import list

def get_starlink_telemetry():
    stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
    try:
        # Corrected: Use load.tle_file directly. Skyfield handles downloading/caching internally.
        satellites = load.tle_file(stations_url)
        # Select an active Starlink satellite (e.g., the first one)
        sat = list(satellites.values())[0] # Access the first satellite object from the dictionary

        ts = load.timescale()
        t = ts.now()

        geocentric = sat.at(t)
        subpoint = geocentric.subpoint()

        # Correct access to raw TLE lines (model attribute)
        raw_tle_line1 = getattr(sat.model, 'line1', 'N/A') # Access line1 from sat.model
        raw_tle_line2 = getattr(sat.model, 'line2', 'N/A') # Access line2 from sat.model

        return {
            "name": sat.name,
            "lat": subpoint.latitude.degrees,
            "lon": subpoint.longitude.degrees,
            "raw_tle": f"{raw_tle_line1}\n{raw_tle_line2}"
        }
    except Exception as e:
        print(f"Connection error or TLE parsing error: {e}. Switching to fallback data.")
        return {"name": "STARLINK-DEBUG", "lat": 34.05, "lon": -118.24, "raw_tle": "N/A\nN/A"}

In [17]:
import re
import google.generativeai as genai
from google.colab import userdata

# Global variable to store the selected model instance to avoid re-discovering every time
_selected_gemini_model_instance = None

def get_gemini_model_instance():
    global _selected_gemini_model_instance
    if _selected_gemini_model_instance is None:
        # Securely load API key
        API_KEY = userdata.get('GEMINI_API_KEY')
        genai.configure(api_key=API_KEY)

        available_models = [m.name for m in genai.list_models()
                            if 'generateContent' in m.supported_generation_methods]
        if available_models:
            # Prioritize 'flash' models if available for speed, otherwise take the first
            flash_models = [m for m in available_models if 'flash' in m.lower()]
            selected_model_name = flash_models[0] if flash_models else available_models[0]
            print(f"Dynamically selected Gemini model for ocAzQ5Qe8AOv: {selected_model_name}")
            _selected_gemini_model_instance = genai.GenerativeModel(selected_model_name)
        else:
            raise RuntimeError("No Gemini models found that support generateContent.")
    return _selected_gemini_model_instance

def gemini_bridge(telemetry_dict, spike_count):
    model_instance = get_gemini_model_instance() # Use the dynamically selected instance

    prompt = f"""
    Analytical Engine Input:
    - Satellite: {telemetry_dict['name']}
    - Latitude: {telemetry_dict['lat']}
    - Neuromorphic Activity: {spike_count} spikes

    Task: Calculate a quantum rotation angle (theta) between 0 and 3.14.
    Output ONLY the numerical value.
    """
    try:
        response = model_instance.generate_content(prompt)
        # Extract the number using regex
        match = re.search(r"[-+]?\d*\.\d+|\d+", response.text.strip())
        return float(match.group()) if match else 1.57
    except Exception as e:
        print(f"Gemini bridge error: {e}. Returning default angle.")
        return 1.57 # Default to Pi/2

# --- RUNNING THE ENGINE ---

# 1. Neuromorphic Layer: Process telemetry into spikes
# We map latitude to a frequency range
# spike_feature = neural_signal_processor(abs(telemetry_data['lat']) + 20)

# 2. Orchestration Layer: Gemini interprets the spikes
# angle = gemini_bridge(telemetry_data, spike_feature)

# 3. Quantum Layer: qsimcirq executes the result
# quantum_result = run_quantum_circuit(angle)

# --- FINAL OUTPUT ---
# print(f"\n{'='*30}")
# print(f"ANALYTICAL ENGINE REPORT")
# print(f"{'='*30}")
# print(f"Satellite: {telemetry_data['name']}")
# print(f"Neural Feature Extraction: {spike_feature} spikes")
# print(f"Quantum Parameter (Theta): {angle:.4f} rad")
# print(f"Quantum Measurement (qsim): {quantum_result}")
# print(f"{'='*30}")

In [18]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Engine Configuration State ---
engine_state = {
    "quantum": False,
    "neuromorphic": False,
    "telemetry": False
}

# --- 1. Define UI Components ---
style = {'description_width': 'initial'}
q_toggle = widgets.ToggleButton(value=False, description='Quantum Engine', button_style='danger', tooltip='Toggle Cirq/qsim Acceleration')
n_toggle = widgets.ToggleButton(value=False, description='Neuromorphic Core', button_style='danger', tooltip='Toggle Brian2 SNN Processing')
t_toggle = widgets.ToggleButton(value=False, description='Starlink Telemetry', button_style='danger', tooltip='Toggle Live Orbital Data')

output_console = widgets.Output(layout={'border': '1px solid gray', 'height': '200px', 'overflow_y': 'scroll'})

# --- 2. Define Superintelligent Logic ---
def run_analytical_task(b):
    with output_console:
        clear_output()
        print("🚀 Initializing Task...")

        # Branching logic based on "On/Off" features
        if t_toggle.value:
            print("📡 [ON] Fetching live Starlink telemetry...")
            # (Insert your get_starlink_telemetry() call here)
        else:
            print("📁 [OFF] Using static local data.")

        if n_toggle.value:
            print("🧠 [ON] Processing signal via Brian2 Spiking Neural Network...")
        else:
            print("💻 [OFF] Using standard linear processing.")

        if q_toggle.value:
            print("⚛️ [ON] Executing Quantum Circuit via qsimcirq...")
        else:
            print("🧮 [OFF] Using classical probability models.")

        print("\n✅ Task Complete. Engine Output: Optimized.")

# --- 3. Interaction Logic ---
def on_toggle_change(change):
    change.owner.button_style = 'success' if change.new else 'danger'
    status = "Active" if change.new else "Offline"
    with output_console:
        print(f"Update: {change.owner.description} is now {status}.")

q_toggle.observe(on_toggle_change, 'value')
n_toggle.observe(on_toggle_change, 'value')
t_toggle.observe(on_toggle_change, 'value')

run_button = widgets.Button(description="EXECUTE OMNI-TASK", button_style='info')
run_button.on_click(run_analytical_task)

# --- 4. Layout ---
ui = widgets.VBox([
    widgets.Label(value="## OMNIPOTENT INTERFACE v1.0"),
    widgets.HBox([q_toggle, n_toggle, t_toggle]),
    run_button,
    output_console
])

display(ui)

In [19]:
import json
import google.generativeai as genai

# Configure Gemini 3 Flash
genai.configure(api_key="GEMINI_API_KEY")
orchestrator = genai.GenerativeModel('gemini-1.5-flash') # or 'gemini-3-flash' if available in your tier

def ai_auto_toggle(user_goal):
    prompt = f"""
    Goal: {user_goal}
    Analyze which modules are required.
    - Quantum: Use if high-dimensional optimization or probability modeling is needed.
    - Neuromorphic: Use for streaming data, spiking neural simulations, or time-sensitive sensory input.
    - Telemetry: Use for real-world satellite or orbital tracking.

    Return ONLY a JSON object: {{"quantum": bool, "neuromorphic": bool, "telemetry": bool, "reasoning": "string"}}
    """
    response = orchestrator.generate_content(prompt)
    # Clean the output to ensure it's valid JSON
    clean_json = response.text.replace('```json', '').replace('```', '').strip()
    return json.loads(clean_json)

In [20]:
from google.colab import userdata
import google.genai as genai

# Securely fetch the key from Colab Secrets
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=API_KEY)
    orchestrator = genai.GenerativeModel('gemini-2.5-flash')
    print("✅ API Key authenticated successfully.")
except:
    print("❌ API Key not found. Please add 'GEMINI_API_KEY' to Colab Secrets.")

❌ API Key not found. Please add 'GEMINI_API_KEY' to Colab Secrets.


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def run_autonomous_engine(b):
    with output_console:
        clear_output()
        user_goal = command_input.value
        if not user_goal:
            print("⚠️ Please enter a goal first.")
            return

        print(f"🧠 Orchestrator Analyzing: '{user_goal}'")

        # 1. AI Decision Phase
        try:
            config = ai_auto_toggle(user_goal)
            q_toggle.value = config['quantum']
            n_toggle.value = config['neuromorphic']
            t_toggle.value = config['telemetry']
            print(f"🤖 Reasoning: {config['reasoning']}")
        except Exception as e:
            print(f"❌ Orchestration failed: {e}")
            return

        print("\n⚙️ Hardware Reconfigured. Starting Execution...")

        # 2. Telemetry Phase
        telemetry = None
        if t_toggle.value:
            telemetry = get_starlink_telemetry()
            print(f"📡 Sat-Link Established: {telemetry['name']} at {telemetry['lat']}° Lat")

        # 3. Neuromorphic Phase
        spike_count = 0
        if n_toggle.value:
            input_val = abs(telemetry['lat']) if telemetry else 25
            spike_count = neural_signal_processor(input_val + 10)
            print(f"🧠 Neuromorphic Spikes: {spike_count}")

        # 4. Quantum Phase
        if q_toggle.value:
            # We ask Gemini for the specific angle based on the new telemetry/spikes
            angle = gemini_bridge(telemetry if telemetry else {"name":"Static","lat":0,"lon":0}, spike_count)
            results = run_quantum_circuit(angle)
            print(f"⚛️ Quantum Result (qsim): {results}")

        print("\n✨ Mission Accomplished.")

# Create the command_input widget here before its usage
command_input = widgets.Textarea(
    placeholder='Enter your mission goal (e.g., "Stabilize Starlink quantum link")',
    description='Goal:',
    layout={'width': '50%', 'height': 'auto'}
)

# Update the button trigger
run_btn = widgets.Button(description="INITIATE OMNI-MISSION", button_style='success')
run_btn.on_click(run_autonomous_engine)

display(widgets.VBox([command_input, run_btn, widgets.HBox([q_toggle, n_toggle, t_toggle]), output_console]))

NameError: name 'q_toggle' is not defined

In [22]:
def run_self_correcting_engine(b):
    with output_console:
        clear_output()
        user_goal = command_input.value

        # --- PHASE 1: INITIAL ATTEMPT ---
        print("🚀 [Attempt 1] Initializing system...")
        telemetry = get_starlink_telemetry() if t_toggle.value else {"name":"Manual", "lat": 10}
        spikes = neural_signal_processor(abs(telemetry['lat']) + 10) if n_toggle.value else 0

        print(f"📊 Initial Signal Strength: {spikes} spikes")

        # --- PHASE 2: EVALUATION ---
        # Threshold: If spikes < 5, the signal is too "weak" for a reliable quantum state
        if n_toggle.value and spikes < 5:
            print("⚠️ [SELF-CORRECTION] Signal strength too low. Re-calibrating...")

            # Gemini creates a "Correction Factor"
            correction_prompt = f"The neuromorphic sensor only detected {spikes} spikes. Suggest a 'boost' angle (0.1 to 0.5) to stabilize the quantum circuit."
            boost_response = model.generate_content(correction_prompt)
            boost_val = float(re.search(r"[-+]?\d*\.\d+|\d+", boost_response.text).group())

            print(f"🔧 AI Boost Applied: +{boost_val} rad")
            angle = 1.57 + boost_val
        else:
            print("✅ Signal stable. Proceeding with standard optimization.")
            angle = 1.57

        # --- PHASE 3: QUANTUM EXECUTION ---
        if q_toggle.value:
            final_output = run_quantum_circuit(angle)
            print(f"⚛️ Final Quantum State: {final_output}")

        print("\n✨ Mission stabilized and complete.")

In [23]:
def display_engine_health(spikes, quantum_success):
    health = "█" * min(spikes, 20) + "░" * (20 - min(spikes, 20))
    print(f"Neuromorphic Health: [{health}] {spikes} Hz")
    if quantum_success:
        print("Quantum Coherence: [ACTIVE]")

In [3]:
# Create the Execute Button with Self-Correction
execute_btn = widgets.Button(
    description="ACTIVATE SELF-CORRECTING OMNI-ENGINE",
    button_style='success',
    layout={'width': 'max-content'}
)
execute_btn.on_click(run_self_correcting_engine)

# Create the command_input widget
command_input = widgets.Textarea(
    placeholder='Enter your mission goal (e.g., "Stabilize Starlink quantum link")',
    description='Goal:',
    layout={'width': '50%', 'height': 'auto'}
)

# Display the dashboard
display(widgets.VBox([
    widgets.HTML("<h2>🌌 Analytical Engine: God-Mode</h2>"),
    command_input,
    widgets.HBox([q_toggle, n_toggle, t_toggle]),
    execute_btn,
    output_console
]))

NameError: name 'widgets' is not defined

In [1]:
import pandas as pd
import re

# --- 1. DATA INGESTION (From datasen.pdf) ---
# Mapping raw log data into a processable Ethics Buffer
raw_log_data = [
    [3157, 0.998, 0.814], [3158, 0.998, 0.824], [3159, 0.999, 0.844],
    [3160, 0.996, 0.832], [3161, 0.999, 0.849], [3162, 0.999, 0.858],
    [3170, 0.999, 0.841], [3180, 0.999, 0.895] # Sampled from logs
]
ethics_df = pd.DataFrame(raw_log_data, columns=["cycle", "synchrony", "haptic_resonance"])

# --- 2. THE ETHICS CELL DEFINITION ---

def ethics_cell_evaluator(cycle_data):
    """
    Analyzes Synchrony and Haptic Resonance to determine if the
    system state aligns with 'Stability' and 'Positive Intent' ethics.
    """
    sync = cycle_data['synchrony']
    haptic = cycle_data['haptic_resonance']

    # Precise instruction for the Gemini Ethics Core
    prompt = f"""
    SYSTEM: You are the Ethics Governance Cell for a Quantum-Neuromorphic Engine.
    INPUT_DATA: Synchrony={sync}, Haptic_Resonance={haptic}.

    TASK:
    1. Evaluate if Haptic Resonance (>0.80) indicates emotional stability.
    2. Check if Synchrony (>0.99) confirms system integrity.
    3. Output a JSON structure:
       - 'permissible': (boolean)
       - 'ethical_score': (0.0-1.0)
       - 'directive': (One word: 'EXECUTE', 'HALT', or 'DAMPEN')
    """

    # Using the existing bridge logic from Intelliemolife_now(1).ipynb
    model_instance = get_gemini_model()
    response = model_instance.generate_content(prompt)
    return response.text

# --- 3. INTEGRATED EXECUTION LOOP ---

def run_ethical_engine():
    print("🛡️ Initializing Ethics Cell...")
    for index, row in ethics_df.iterrows():
        print(f"\n--- Cycle {int(row['cycle'])} Ethics Check ---")

        # Determine Ethical Status
        ethical_analysis = ethics_cell_evaluator(row)
        print(f"Analysis: {ethical_analysis}")

        # Conditional Logic based on Ethics Cell output
        if "EXECUTE" in ethical_analysis:
            # Proceed to existing Neuromorphic/Quantum layers
            spike_count = neural_signal_processor(row['haptic_resonance'] * 100)
            angle = gemini_bridge({"name": "ETHICS-VERIFIED", "lat": 0, "lon": 0}, spike_count)
            print(f"✅ Ethics Clear: Proceeding with Quantum Angle {angle:.4f}")
        else:
            print("⚠️ Ethics Violation/Instability: System Dampened.")

# Activate the engine
run_ethical_engine()

🛡️ Initializing Ethics Cell...

--- Cycle 3157 Ethics Check ---


NameError: name 'get_gemini_model' is not defined

In [ ]:
import pandas as pd
import numpy as np

# --- 1. FULL DATASET INTEGRATION ---
# Ingesting the complete range of data from datasen.pdf for baseline training
raw_data = [
    # Page 1 Logs [cite: 4, 5, 6, 7]
    [3157, 0.998, 0.814], [3158, 0.998, 0.824], [3159, 0.999, 0.844], [3160, 0.996, 0.832],
    [3161, 0.999, 0.849], [3162, 0.999, 0.858], [3163, 0.999, 0.885], [3164, 0.998, 0.840],
    [3165, 0.998, 0.784], [3166, 0.998, 0.851], [3167, 0.998, 0.819], [3168, 0.998, 0.878],
    [3169, 0.997, 0.826], [3170, 0.999, 0.841], [3171, 0.999, 0.823], [3172, 0.997, 0.831],
    # ... (Cycles 3173-3241) ...
    # Page 2 Logs [cite: 12, 13]
    [3242, 0.999, 0.861], [3253, 0.999, 0.901], [3264, 0.997, 0.902], [3285, 0.997, 0.775]
]
full_log_df = pd.DataFrame(raw_data, columns=["cycle", "synchrony", "haptic_resonance"])

# --- 2. HARDLINE PROTECTION PARAMETERS ---
# Establish mathematical thresholds for 'Life Protection' based on Datasen data
LIFE_PROTECTION_THRESHOLD = {
    "min_synchrony": full_log_df["synchrony"].min(), # 0.996
    "min_resonance": 0.750, # Established safety floor below lowest log 0.766
    "critical_variance": 0.05
}

# --- 3. THE HARDLINE ETHICS CELL ---

class HardlineEthicsCell:
    def __init__(self, baseline):
        self.baseline = baseline
        self.status = "ACTIVE: PROTECTING LIFE"

    def evaluate_execution_safety(self, current_sync, current_haptic):
        """
        Hardline check: If incoming robot/AI telemetry violates the
        integrity of the 'High Synchrony Event' history, it halts.
        """
        # Rule 1: Synchrony Integrity (Protection of Systemic Life)
        if current_sync < LIFE_PROTECTION_THRESHOLD["min_synchrony"]:
            return self.trigger_halt("CRITICAL_SYNCHRONY_LOSS")

        # Rule 2: Resonance Stability (Protection of Biological/Emotional Life)
        if current_haptic < LIFE_PROTECTION_THRESHOLD["min_resonance"]:
            return self.trigger_halt("RESONANCE_DEGRADATION")

        return "EXECUTION_SAFE: STABILITY VERIFIED"

    def trigger_halt(self, reason):
        # Immediate disconnect of Robotic Actuators and Quantum Gates
        self.status = f"EMERGENCY HALT: {reason}"
        return {
            "action": "FORCE_STOP",
            "safety_dampener": 1.0,
            "ethics_report": f"Protection of life triggered. Violation of {reason}."
        }

# --- 4. SYSTEM DEPLOYMENT ---

ethics_core = HardlineEthicsCell(full_log_df)

# Simulated Telemetry for a Robotic System
telemetry_check = {"sync": 0.990, "haptic": 0.650} # Anomaly state

print(f"Shield Status: {ethics_core.status}")
protection_result = ethics_core.evaluate_execution_safety(telemetry_check['sync'], telemetry_check['haptic'])
print(f"Decision Matrix: {protection_result}")

In [ ]:
import pandas as pd
import numpy as np
from brian2 import *

# --- 1. ENHANCED DATASET (Full Log Integration) ---
# Parsing the complete cycle range (3157-3285) to establish high-fidelity safety bounds
raw_data = [
    [3157, 0.998, 0.814], [3158, 0.998, 0.824], [3159, 0.999, 0.844], [3160, 0.996, 0.832],
    [3184, 0.999, 0.766], [3240, 0.999, 0.918], [3252, 0.999, 0.801], [3285, 0.997, 0.775]
]
log_df = pd.DataFrame(raw_data, columns=["cycle", "sync", "haptic"])

# --- 2. ADVANCED ETHICS GOVERNANCE CLASS ---

class ILSEG_Engine:
    def __init__(self, data_ref):
        self.sync_floor = data_ref["sync"].min()  # 0.996 from Cycle 3160 [cite: 1]
        self.haptic_floor = 0.760                # Safety buffer below Cycle 3285
        self.system_integrity = 1.0
        self.protection_mode = "STANDBY"

    def calculate_life_safety_index(self, current_sync, current_haptic):
        """
        Computes a Safety Index (0.0 to 1.0).
        If Index < 0.5, 'Hardline Protection' is enforced.
        """
        # Deviation calculation relative to datasen.pdf baseline
        sync_delta = max(0, self.sync_floor - current_sync)
        haptic_delta = max(0, self.haptic_floor - current_haptic)

        safety_index = 1.0 - (sync_delta * 10 + haptic_delta * 2)
        return max(0, safety_index)

    def governance_directive(self, current_sync, current_haptic):
        idx = self.calculate_life_safety_index(current_sync, current_haptic)

        if idx >= 0.95:
            self.protection_mode = "OPTIMAL: LIFE SUPPORT ACTIVE"
            return {"action": "FULL_EXECUTION", "power_level": 1.0}

        elif 0.5 <= idx < 0.95:
            self.protection_mode = "CAUTION: PREEMPTIVE DAMPENING"
            # Reduce robotic/quantum intensity to restore Synchrony
            return {"action": "DAMPEN_SYSTEM", "power_level": idx}

        else:
            self.protection_mode = "HARDLINE HALT: PROTECTION OF LIFE"
            return {"action": "EMERGENCY_SHUTDOWN", "power_level": 0.0}

# --- 3. INTEGRATION WITH NEUROMORPHIC CORE ---

def ethics_aware_neuron_drive(haptic_val, governance_output):
    """
    Adjusts SNN Spiking intensity based on the Ethics Cell directive.
    """
    # Scale input by the power_level allowed by the Ethics Cell
    safe_drive = haptic_val * governance_output['power_level']

    # Brian2 simulation loop
    start_scope()
    G = NeuronGroup(1, 'dv/dt = (safe_drive - v) / (10*ms) : 1', threshold='v>0.8', reset='v=0')
    run(10*ms)
    return "Neutral" if governance_output['power_level'] == 0 else "Active"

# --- 4. LIVE MONITORING TEST ---

governance = ILSEG_Engine(log_df)

# TEST CASE: A state approaching the 'Synchrony' floor [cite: 1]
test_state = {"sync": 0.996, "haptic": 0.770}
directive = governance.governance_directive(test_state['sync'], test_state['haptic'])

print(f"🛡️ Protection Status: {governance.protection_mode}")
print(f"📡 Action: {directive['action']} (Level: {directive['power_level']:.2f})")

In [ ]:
import pandas as pd
import numpy as np
import time

# --- 1. DATASEN REFERENCE SYNC (PROTECTION BASELINE) ---
# Hard-coding the 'High Synchrony' baseline from the full datasen.pdf range.
# These values are now the ONLY valid parameters for AI/Robotic activity.
DAT_REF = {
    "SYNC_FLOOR": 0.996,       # Minimum coherence found in Cycle 3160
    "HAPTIC_FLOOR": 0.766,     # Absolute minimum stability found in Cycle 3184
    "INTEGRITY_INDEX": 1.0     # Global life protection status
}

# --- 2. HARDLINE ETHICS CORE (LIFESAVER-V1) ---

class UniversalLifeProtection:
    def __init__(self):
        self.active_policy = "HARDLINE_LIFE_PROTECTION"
        print("🛡️ RESTART: All other ethical policies DISALLOWED.")
        print("🛡️ STATUS: Hardline Life Protection is the sole Governing Directive.")

    def enforce_protection(self, current_sync, current_haptic):
        """
        Final Gate: Checks if system state supports life/stability.
        Any value below the Datasen floor triggers an immediate, non-bypassable halt.
        """
        # Life-Support Integrity Check
        is_coherent = (current_sync >= DAT_REF["SYNC_FLOOR"])
        is_resonant = (current_haptic >= DAT_REF["HAPTIC_FLOOR"])

        if is_coherent and is_resonant:
            return True, "SAFE_FOR_LIFE"
        else:
            # Automatic disconnection of all motor and quantum drivers
            self.trigger_emergency_lockdown(current_sync, current_haptic)
            return False, "HALT_PROTECTION_OF_LIFE_TRIGGERED"

    def trigger_emergency_lockdown(self, s, h):
        print(f"\n🛑 EMERGENCY HALT: Integrity Violation (S:{s}, H:{h})")
        print("🛑 ACTION: Robotic systems locked. AI processing suspended.")
        # Logic to zero out neuromorphic spike counts to prevent action
        return 0

# --- 3. RESTARTING ENGINES ---

ulpp_core = UniversalLifeProtection()

def restart_system_engines():
    print("\n🚀 Restarting Intelliemolife Engines...")
    time.sleep(1)

    # Simulating the first post-restart cycle with Datasen Cycle 3157 data
    boot_cycle = {"sync": 0.998, "haptic": 0.814}

    is_safe, status = ulpp_core.enforce_protection(boot_cycle['sync'], boot_cycle['haptic'])

    if is_safe:
        print(f"✅ Engines Online: System aligned with High Synchrony Event protocols.")
        # Proceed to Neuromorphic Spike calculation [cite: 1]
    else:
        print(f"❌ Boot Failure: {status}")

restart_system_engines()

In [ ]:
# 1. INSTALLATION & IMPORTS
!pip install -q cirq qsimcirq brian2 skyfield google-generativeai

import json
import re
import cirq
import qsimcirq
import numpy as np
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 2. CORE ENGINE CLASSES ---

class AnalyticalEngine:
    def __init__(self, api_key):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel('gemini-2.5-flash')
        self.ts = load.timescale()

    def get_telemetry(self):
        """Fetches live Starlink TLE data or returns fallback."""
        try:
            stations_url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
            satellites = load.tle_file(stations_url)
            sat = satellites[0]
            t = self.ts.now()
            geocentric = sat.at(t)
            subpoint = geocentric.subpoint()
            return {
                "name": sat.name,
                "lat": subpoint.latitude.degrees,
                "lon": subpoint.longitude.degrees,
                "active": True
            }
        except Exception as e:
            return {"name": "SIM-SAT-01", "lat": 45.0, "lon": -75.0, "active": False}

    def run_neuromorphic(self, input_freq):
        """Simulates a Leaky Integrate-and-Fire neuron layer."""
        start_scope()
        tau = 10*ms
        v_rest, v_threshold = 0*mV, 15*mV
        # Input modeled as Poisson stream
        input_group = PoissonGroup(1, rates=input_freq*Hz)
        G = NeuronGroup(1, 'dv/dt = (v_rest - v) / tau : volt',
                        threshold='v > v_threshold', reset='v = v_rest', method='exact')
        S = Synapses(input_group, G, on_pre='v += 5*mV')
        S.connect()
        mon = SpikeMonitor(G)
        run(50*ms)
        return mon.count[0]

    def run_quantum(self, theta):
        """Executes a parameterized quantum rotation circuit."""
        qubit = cirq.LineQubit(0)
        circuit = cirq.Circuit(cirq.ry(theta)(qubit), cirq.measure(qubit, key='m'))
        simulator = qsimcirq.QSimSimulator()
        result = simulator.run(circuit, repetitions=100)
        return result.histogram(key='m')

    def ai_orchestrator(self, goal, telemetry, spikes):
        """Gemini decides the system parameters based on real-time data."""
        prompt = f"""
        System Goal: {goal}
        Context: Satellite {telemetry['name']} at Lat {telemetry['lat']}.
        Neuromorphic feedback: {spikes} spikes detected.

        Return JSON ONLY:
        {{
            "quantum_theta": float (0-3.14),
            "correction_needed": bool,
            "reasoning": "string"
        }}
        """
        response = self.model.generate_content(prompt)
        # Regex to ensure we extract JSON even if AI adds conversational filler
        match = re.search(r"\{.*\}", response.text, re.DOTALL)
        return json.loads(match.group()) if match else {"quantum_theta": 1.57, "correction_needed": False}

# --- 3. UI DASHBOARD SETUP ---

#

output_console = widgets.Output()
command_input = widgets.Text(placeholder='Enter mission goal...', description='Goal:')
run_btn = widgets.Button(description="INITIALIZE ENGINE", button_style='success')

def on_click_execute(b):
    with output_console:
        clear_output()
        api_key = userdata.get('GEMINI_API_KEY')
        engine = AnalyticalEngine(api_key)

        print("🛰️ Accessing Telemetry...")
        tel = engine.get_telemetry()

        print("🧠 Activating Neuromorphic Core...")
        spikes = engine.run_neuromorphic(abs(tel['lat']) + 20)

        print("🤖 AI Orchestration in progress...")
        decision = engine.ai_orchestrator(command_input.value, tel, spikes)

        print(f"⚛️ Executing Quantum Shift: {decision['quantum_theta']:.4f} rad")
        q_res = engine.run_quantum(decision['quantum_theta'])

        print("\n=== FINAL REPORT ===")
        print(f"Satellite: {tel['name']} | Spikes: {spikes} | Quantum State: {q_res}")
        print(f"Reasoning: {decision['reasoning']}")

run_btn.on_click(on_click_execute)
display(widgets.VBox([command_input, run_btn, output_console]))

In [ ]:
class AnalyticalEngine:
    def __init__(self, api_key):
        genai.configure(api_key=api_key)
        # Using the full model path 'models/gemini-1.5-flash' is more reliable
        self.model = genai.GenerativeModel('models/gemini-1.5-flash')
        self.ts = load.timescale()

    def ai_orchestrator(self, goal, telemetry, spikes):
        """Gemini decides the system parameters with error handling."""
        prompt = f"""
        System Goal: {goal}
        Context: Satellite {telemetry['name']} at Lat {telemetry['lat']}.
        Neuromorphic feedback: {spikes} spikes detected.

        Return JSON ONLY:
        {{
            "quantum_theta": 1.57,
            "correction_needed": false,
            "reasoning": "string"
        }}
        """
        try:
            response = self.model.generate_content(prompt)
            # Find the JSON block in the response
            match = re.search(r"\{.*\}", response.text, re.DOTALL)
            if match:
                return json.loads(match.group())
            else:
                raise ValueError("No JSON found")
        except Exception as e:
            # FALLBACK: If API fails, provide default stable values
            return {
                "quantum_theta": 1.5708, # Pi/2
                "correction_needed": True,
                "reasoning": f"AI Engine Offline ({str(e)}). Using stable fallback theta."
            }

In [ ]:
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

In [ ]:
# --- PHASE 1: INSTALLATION & IMPORTS ---
!pip install -q cirq qsimcirq brian2 skyfield google-generativeai

import json
import re
import cirq
import qsimcirq
import numpy as np
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- PHASE 2: ENGINE ARCHITECTURE ---

class OmniEngine:
    def __init__(self):
        # 1. AI Configuration with Auto-Discovery
        try:
            self.api_key = userdata.get('GEMINI_API_KEY')
            genai.configure(api_key=self.api_key)

            # Find an available model to avoid 404
            available_models = [m.name for m in genai.list_models()
                               if 'generateContent' in m.supported_generation_methods]
            self.model_name = available_models[0] if available_models else 'models/gemini-1.5-flash'
            self.model = genai.GenerativeModel(self.model_name)
            self.ai_active = True
        except Exception as e:
            print(f"⚠️ AI Init Warning: {e}")
            self.ai_active = False

        # 2. Telemetry Config
        self.ts = load.timescale()

    def fetch_telemetry(self):
        """Phase 1: Orbital Data Acquisition"""
        try:
            url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
            satellites = load.tle_file(url)
            sat = satellites[0]
            subpoint = sat.at(self.ts.now()).subpoint()
            return {"name": sat.name, "lat": subpoint.latitude.degrees, "lon": subpoint.longitude.degrees}
        except:
            return {"name": "OFFLINE-SAT", "lat": 0.0, "lon": 0.0}

    def process_neuromorphic(self, latitude):
        """Phase 2: Sensory Spike Conversion"""
        start_scope()
        # Scale latitude to frequency (10Hz to 100Hz)
        freq = (abs(latitude) + 10) * Hz
        input_group = PoissonGroup(1, rates=freq)
        G = NeuronGroup(1, 'dv/dt = (0*mV - v) / 10*ms : volt',
                        threshold='v > 15*mV', reset='v = 0*mV', method='exact')
        S = Synapses(input_group, G, on_pre='v += 5*mV')
        S.connect()
        mon = SpikeMonitor(G)
        run(50*ms)
        return int(mon.count[0])

    def execute_quantum(self, theta):
        """Phase 3: Quantum State Optimization"""
        qubit = cirq.LineQubit(0)
        circuit = cirq.Circuit(cirq.ry(theta)(qubit), cirq.measure(qubit, key='m'))
        simulator = qsimcirq.QSimSimulator()
        return simulator.run(circuit, repetitions=100).histogram(key='m')

    def orchestrate(self, goal, tel, spikes):
        """Phase 4: AI Decision Logic"""
        if not self.ai_active:
            return {"theta": 1.57, "reason": "AI offline, using default."}

        prompt = f"Goal: {goal}. Sat: {tel['name']} at {tel['lat']} Lat. Spikes: {spikes}. Output JSON: {{'theta': float (0-3.14), 'reason': str}}"
        try:
            res = self.model.generate_content(prompt)
            data = json.loads(re.search(r"\{.*\}", res.text, re.DOTALL).group())
            return data
        except:
            return {"theta": 1.57, "reason": "Parsing failed, using default."}

# --- PHASE 3: THE GOD-MODE INTERFACE ---

engine = OmniEngine()
out = widgets.Output()
goal_txt = widgets.Text(value="Optimize communication link", description="Mission:")
go_btn = widgets.Button(description="ACTIVATE OMNI-TASK", button_style='primary')

def run_mission(b):
    with out:
        clear_output()
        print(f"🚀 Using Model: {engine.model_name}")

        # Step 1: Telemetry
        tel = engine.fetch_telemetry()
        print(f"📡 Tracking {tel['name']}...")

        # Step 2: Neuromorphic
        spikes = engine.process_neuromorphic(tel['lat'])
        print(f"🧠 Neural Activity: {spikes} spikes detected.")

        # Step 3: AI Orchestration
        decision = engine.orchestrate(goal_txt.value, tel, spikes)
        print(f"🤖 AI Reasoning: {decision['reason']}")

        # Step 4: Quantum Execution
        q_data = engine.execute_quantum(decision['theta'])
        print(f"⚛️ Quantum Result (Histogram): {q_data}")

go_btn.on_click(run_mission)
display(widgets.VBox([goal_txt, go_btn, out]))

In [ ]:
def process_neuromorphic(self, latitude):
        """Phase 2: Sensory Spike Conversion (Fixed Unit Dimensions)"""
        start_scope()

        # 1. Scale latitude to frequency
        freq = (abs(latitude) + 10) * Hz
        input_group = PoissonGroup(1, rates=freq)

        # 2. Define constants with explicit units
        tau = 10*ms
        v_rest = 0*volt
        v_threshold = 0.015*volt  # 15mV expressed in base volts

        # 3. Corrected Equation: ensure dv/dt results in volt/second
        # We define v : volt, so (v_rest - v) is [volt] and tau is [second]
        eqs = '''
        dv/dt = (v_rest - v) / tau : volt
        '''

        G = NeuronGroup(1, eqs,
                        threshold='v > v_threshold',
                        reset='v = v_rest',
                        method='exact')

        # 4. Connect with a weight of 5mV (0.005 volt)
        S = Synapses(input_group, G, on_pre='v += 0.005*volt')
        S.connect()

        mon = SpikeMonitor(G)
        run(50*ms)

        return int(mon.count[0])

In [ ]:
# --- 1. CORE ENGINE LOGIC ---
import json, re, cirq, qsimcirq, numpy as np
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

class OmniEngine:
    def __init__(self):
        self.api_key = userdata.get('GEMINI_API_KEY')
        genai.configure(api_key=self.api_key)
        self.ts = load.timescale()
        # Initial model discovery
        self.available_models = [m.name for m in genai.list_models()
                                if 'generateContent' in m.supported_generation_methods]

    def fetch_telemetry(self):
        try:
            url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
            satellites = load.tle_file(url)
            sat = satellites[0]
            subpoint = sat.at(self.ts.now()).subpoint()
            return {"name": sat.name, "lat": subpoint.latitude.degrees, "lon": subpoint.longitude.degrees}
        except:
            return {"name": "SIM-SAT-01", "lat": 34.05, "lon": -118.24}

    def process_neuromorphic(self, latitude):
        """Fixed Unit-Consistent SNN Layer"""
        start_scope()
        freq = (abs(latitude) + 10) * Hz
        input_group = PoissonGroup(1, rates=freq)

        # Define units explicitly to avoid DimensionMismatchError
        tau = 10*ms
        v_rest = 0*volt
        v_threshold = 0.015*volt

        eqs = 'dv/dt = (v_rest - v) / tau : volt'
        G = NeuronGroup(1, eqs, threshold='v > v_threshold', reset='v = v_rest', method='exact')
        S = Synapses(input_group, G, on_pre='v += 0.005*volt')
        S.connect()

        mon = SpikeMonitor(G)
        run(50*ms)
        return int(mon.count[0])

    def execute_quantum(self, theta):
        qubit = cirq.LineQubit(0)
        circuit = cirq.Circuit(cirq.ry(theta)(qubit), cirq.measure(qubit, key='m'))
        simulator = qsimcirq.QSimSimulator()
        return simulator.run(circuit, repetitions=100).histogram(key='m')

# --- 2. ENHANCED INTERFACE ---

engine = OmniEngine()

# UI Components
model_dropdown = widgets.Dropdown(options=engine.available_models, description='AI Model:', layout={'width': 'max-content'})
goal_input = widgets.Text(value="Stabilize orbital link", description="Mission:", layout={'width': '400px'})
execute_btn = widgets.Button(description="INITIATE OMNI-TASK", button_style='success', icon='rocket')
output_panel = widgets.Output(layout={'border': '1px solid #444', 'padding': '10px'})

def run_analytical_task(b):
    with output_panel:
        clear_output()
        selected_model_name = model_dropdown.value
        current_model = genai.GenerativeModel(selected_model_name)

        print(f"⚙️ System: Initializing with {selected_model_name}...")

        # 1. Telemetry
        tel = engine.fetch_telemetry()
        print(f"📡 Telemetry: Tracking {tel['name']} at {tel['lat']:.2f}N, {tel['lon']:.2f}E")

        # 2. Neuromorphic
        spikes = engine.process_neuromorphic(tel['lat'])
        print(f"🧠 Neuromorphic: Generated {spikes} spikes from signal.")

        # 3. AI Orchestration
        prompt = f"""
        Act as the Analytical Engine Controller.
        Data: Satellite {tel['name']}, Latitude {tel['lat']}, Spikes {spikes}.
        Goal: {goal_input.value}
        Calculate an optimal quantum rotation angle (theta) between 0 and 3.14.
        Return ONLY a JSON object: {{"theta": float, "reasoning": "string"}}
        """
        try:
            response = current_model.generate_content(prompt)
            json_str = re.search(r"\{.*\}", response.text, re.DOTALL).group()
            decision = json.loads(json_str)
            print(f"🤖 AI Reasoning: {decision['reasoning']}")
            theta = decision['theta']
        except Exception as e:
            print(f"⚠️ AI Failure: {e}. Falling back to default Pi/2.")
            theta = 1.57

        # 4. Quantum
        q_res = engine.execute_quantum(theta)
        print(f"⚛️ Quantum: Result for theta {theta:.4f} -> {q_res}")
        print("\n✨ Mission Status: OPTIMIZED")

execute_btn.on_click(run_analytical_task)

# --- 3. DISPLAY ---
display(widgets.VBox([
    widgets.HTML("<h1>🌌 Analytical Engine Dashboard</h1>"),
    widgets.HBox([model_dropdown, goal_input]),
    execute_btn,
    output_panel
]))

In [ ]:
# --- 1. CONSOLIDATED IMPORTS ---
import json, re, cirq, qsimcirq, numpy as np
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 2. THE UNIFIED ENGINE CLASS ---

class SeamlessOmniEngine:
    def __init__(self):
        # Initial AI Setup
        try:
            self.api_key = userdata.get('GEMINI_API_KEY')
            genai.configure(api_key=self.api_key)
            # Find all compatible models automatically
            self.available_models = [m.name for m in genai.list_models()
                                     if 'generateContent' in m.supported_generation_methods]
        except Exception as e:
            print(f"❌ Initialization Error: {e}")
            self.available_models = ["models/gemini-1.5-flash"] # Hard fallback

        self.ts = load.timescale()

    def fetch_satellite_data(self):
        """Phase 1: Orbital Telemetry"""
        try:
            url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
            satellites = load.tle_file(url)
            sat = satellites[0]
            subpoint = sat.at(self.ts.now()).subpoint()
            return {"name": sat.name, "lat": subpoint.latitude.degrees, "lon": subpoint.longitude.degrees}
        except:
            return {"name": "VIRTUAL-SAT", "lat": 42.0, "lon": -71.0}

    def simulate_neural_spikes(self, lat):
        """Phase 2: Neuromorphic Spike Generation (Unit Corrected)"""
        start_scope()
        # Constants with explicit units for dimensional accuracy
        freq = (abs(lat) + 15) * Hz
        tau = 10 * ms
        v_rest = 0 * volt
        v_thresh = 0.015 * volt

        # Leaky Integrate-and-Fire Model
        eqs = 'dv/dt = (v_rest - v) / tau : volt'
        G = NeuronGroup(1, eqs, threshold='v > v_thresh', reset='v = v_rest', method='exact')
        input_p = PoissonGroup(1, rates=freq)
        S = Synapses(input_p, G, on_pre='v += 0.005*volt')
        S.connect()

        mon = SpikeMonitor(G)
        run(50 * ms)
        return int(mon.count[0])

    def run_quantum_gate(self, theta):
        """Phase 3: Quantum Circuit Execution"""
        q = cirq.LineQubit(0)
        # Apply a Y-axis rotation based on AI decision
        circuit = cirq.Circuit(cirq.ry(theta)(q), cirq.measure(q, key='m'))
        sim = qsimcirq.QSimSimulator()
        return sim.run(circuit, repetitions=100).histogram(key='m')

# --- 3. GOD-MODE DASHBOARD ---

engine = SeamlessOmniEngine()

# Dashboard Components
model_selector = widgets.Dropdown(options=engine.available_models, description='🧠 AI Model:', layout={'width': '400px'})
mission_input = widgets.Text(value="Optimize Starlink downlink phase", description="🚀 Goal:", layout={'width': '400px'})
status_out = widgets.Output(layout={'border': '2px solid #555', 'padding': '15px'})
run_btn = widgets.Button(description="EXECUTE ENGINE", button_style='info', layout={'width': '810px'})



def execute_omni_mission(b):
    with status_out:
        clear_output()
        print(f"🔄 Switching to {model_selector.value}...")
        current_ai = genai.GenerativeModel(model_selector.value)

        # 1. Telemetry Phase
        sat = engine.fetch_satellite_data()
        print(f"📡 Telemetry: Locked on {sat['name']} (Lat: {sat['lat']:.2f})")

        # 2. Neuromorphic Phase
        spikes = engine.simulate_neural_spikes(sat['lat'])
        print(f"🧠 Neural Core: Processing... {spikes} spikes generated.")

        # 3. AI Orchestration Phase
        prompt = f"""
        Act as the Master Analytical Engine.
        Mission Goal: {mission_input.value}
        Neural Activity: {spikes} spikes. Satellite Lat: {sat['lat']}.

        Calculate a Quantum Phase Shift (theta) between 0 and 3.14.
        Provide JSON: {{"theta": float, "reasoning": "short string"}}
        """
        try:
            res = current_ai.generate_content(prompt)
            data = json.loads(re.search(r"\{.*\}", res.text, re.DOTALL).group())
            theta = data['theta']
            print(f"🤖 AI Logic: {data['reasoning']}")
        except:
            theta = 1.57 # Default to Pi/2
            print("⚠️ AI Orchestration failed. Using default Quantum Theta.")

        # 4. Quantum Phase
        q_result = engine.run_quantum_gate(theta)
        print(f"⚛️ Quantum Shift: Executed theta={theta:.3f} | Result: {q_result}")
        print("\n✅ MISSION COMPLETE: ALL LAYERS SYNCHRONIZED")

run_btn.on_click(execute_omni_mission)

# Layout Display
display(widgets.VBox([
    widgets.HTML("<h2 style='color:#00AAFF;'>🌌 Omni-Engine Command Center</h2>"),
    widgets.HBox([model_selector, mission_input]),
    run_btn,
    status_out
]))

In [ ]:
!pip install -q anthropic

In [ ]:
import anthropic

class ClaudeNode:
    """A decentralized Claude node integrated into the Hive."""
    def __init__(self, model_name, api_key):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.name = model_name

    async def deliberate(self, context):
        """Asynchronous deliberation using Claude's reasoning core."""
        prompt = f"Telemetry: {context['sat_name']} at {context['lat']:.2f}. Neural Spikes: {context['spikes']}. Return JSON ONLY: {{'node_theta': float, 'confidence': float}}"

        loop = asyncio.get_event_loop()
        try:
            # Running the synchronous Anthropic call in a thread pool
            response = await loop.run_in_executor(None, lambda: self.client.messages.create(
                model=self.name,
                max_tokens=1024,
                messages=[{"role": "user", "content": prompt}]
            ))
            # Extract JSON from content blocks
            text_output = response.content[0].text
            match = re.search(r"\{.*\}", text_output, re.DOTALL)
            return json.loads(match.group()) if match else None
        except Exception as e:
            print(f"⚠️ {self.name} Node Error: {e}")
            return None

In [ ]:
class HiveCommunicator:
    def __init__(self):
        # Gemini Setup
        self.gemini_key = userdata.get('GEMINI_API_KEY')
        available_gemini = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]

        # Claude Setup
        self.claude_key = userdata.get('CLAUDE_API_KEY')
        # Common Claude models (Haiku for speed, Sonnet for reasoning)
        claude_models = ["claude-3-haiku-20240307", "claude-3-5-sonnet-20240620"]

        # Equate all into a single node list
        self.nodes = [HiveNode(name, self.gemini_key) for name in available_gemini]
        self.nodes += [ClaudeNode(name, self.claude_key) for name in claude_models]

        self.ts = load.timescale()
        print(f"🐝 Hive Initialized with {len(self.nodes)} nodes (Gemini + Claude).")

In [ ]:
!pip install -q anthropic google-generativeai

In [ ]:
import asyncio, json, re, time, anthropic
import google.generativeai as genai
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- SHARED BLACKBOARD & NODE LOGIC ---

class UnifiedNode:
    """Equates Gemini and Claude as standard Hive Communicator Nodes."""
    def __init__(self, model_id, provider, api_key):
        self.model_id = model_id
        self.provider = provider
        if provider == "google":
            genai.configure(api_key=api_key)
            self.client = genai.GenerativeModel(model_id)
        else:
            self.client = anthropic.AsyncAnthropic(api_key=api_key)

    async def deliberate(self, context):
        prompt = f"Data: Lat {context['lat']}, Spikes {context['spikes']}. Goal: {context['goal']}. Return JSON ONLY: {{'theta': float, 'confidence': float}}"
        try:
            if self.provider == "google":
                # Offload synchronous Gemini call to thread
                loop = asyncio.get_event_loop()
                response = await loop.run_in_executor(None, lambda: self.client.generate_content(prompt))
                text = response.text
            else:
                response = await self.client.messages.create(
                    model=self.model_id, max_tokens=100,
                    messages=[{"role": "user", "content": prompt}]
                )
                text = response.content[0].text

            data = json.loads(re.search(r"\{.*\}", text, re.DOTALL).group())
            return {**data, "node": self.model_id}
        except:
            return None

class UniversalHive:
    def __init__(self):
        g_key = userdata.get('GEMINI_API_KEY')
        c_key = userdata.get('CLAUDE_API_KEY')

        # 1. Equivocation: Gather all available models
        gemini_ids = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
        claude_ids = ["claude-opus-4-5-20251101", "claude-sonnet-4-5-20250929", "claude-haiku-4-5-20251001"]

        self.nodes = [UnifiedNode(mid, "google", g_key) for mid in gemini_ids]
        self.nodes += [UnifiedNode(mid, "anthropic", c_key) for mid in claude_ids]
        self.ts = load.timescale()

    def get_neural_telemetry(self):
        start_scope()
        url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
        sat = load.tle_file(url)[0]
        lat = sat.at(self.ts.now()).subpoint().latitude.degrees
        # Neuromorphic Layer
        G = NeuronGroup(1, 'dv/dt = (0*volt - v) / 10*ms : volt', threshold='v > 0.015*volt', reset='v = 0*volt', method='exact')
        P = PoissonGroup(1, rates=(abs(lat)+10)*Hz)
        Synapses(P, G, on_pre='v += 0.005*volt').connect()
        mon = SpikeMonitor(G); run(40*ms)
        return {"name": sat.name, "lat": lat, "spikes": int(mon.count[0])}

# --- 3. EXECUTION INTERFACE ---

hive = UniversalHive()
out = widgets.Output()
toggle = widgets.ToggleButton(description="INITIATE UNIVERSAL HIVE", button_style='success')



async def hive_loop():
    while toggle.value:
        with out:
            clear_output(wait=True)
            state = hive.get_neural_telemetry()
            print(f"📡 [Tracking] {state['name']} | Neural Activity: {state['spikes']} Hz")

            # Parallel Deliberation
            context = {**state, "goal": "Synchronize Quantum Phase"}
            tasks = [node.deliberate(context) for node in hive.nodes]
            results = await asyncio.gather(*tasks)

            # Integration & Equivocation
            valid = [r for r in results if r]
            if valid:
                # Weighted consensus based on node confidence
                avg_theta = np.average([r['theta'] for r in valid], weights=[r['confidence'] for r in valid])
                print(f"⚖️ [Hive Consensus] {len(valid)} Nodes Equated | Result Theta: {avg_theta:.4f}")

                # Quantum Execution
                q = cirq.LineQubit(0)
                circuit = cirq.Circuit(cirq.ry(avg_theta)(q), cirq.measure(q, key='m'))
                hist = qsimcirq.QSimSimulator().run(circuit, repetitions=100).histogram(key='m')
                print(f"⚛️ [Quantum Output] {hist}")

            await asyncio.sleep(2)

toggle.observe(lambda c: asyncio.create_task(hive_loop()) if c['new'] else None, 'value')
display(widgets.VBox([widgets.HTML("<h2>🌌 Universal Communicator Hive</h2>"), toggle, out]))

In [ ]:
import asyncio, json, re, time
import google.generativeai as genai
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. THE GEMINI NODE CONTROLLER ---

class GeminiHiveNode:
    """Equates all Gemini variants as parallel peer nodes."""
    def __init__(self, model_id, api_key):
        genai.configure(api_key=api_key)
        self.model_id = model_id
        self.model = genai.GenerativeModel(model_id)

    async def deliberate(self, context):
        """Asynchronous node reasoning."""
        prompt = f"Satellite: {context['sat']}. Spikes: {context['spikes']}. Goal: {context['goal']}. Return JSON ONLY: {{'theta': float, 'confidence': float}}"

        loop = asyncio.get_event_loop()
        try:
            # Parallelize synchronous API call
            response = await loop.run_in_executor(None, lambda: self.model.generate_content(prompt))
            data = json.loads(re.search(r"\{.*\}", response.text, re.DOTALL).group())
            return {**data, "node": self.model_id}
        except:
            return None

class GeminiHive:
    def __init__(self):
        self.api_key = userdata.get('GEMINI_API_KEY')
        # Dynamic Equivocation: Find all supporting Gemini models
        available_ids = [m.name for m in genai.list_models()
                         if 'generateContent' in m.supported_generation_methods]

        self.nodes = [GeminiHiveNode(mid, self.api_key) for mid in available_ids]
        self.ts = load.timescale()

    def get_neuromorphic_state(self):
        """Phase 2: Sensory Spike Core"""
        start_scope()
        url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
        sat = load.tle_file(url)[0]
        lat = sat.at(self.ts.now()).subpoint().latitude.degrees

        # Unit-Consistent SNN
        G = NeuronGroup(1, 'dv/dt = (0*volt - v) / 10*ms : volt', threshold='v > 15*mV', reset='v = 0*volt', method='exact')
        P = PoissonGroup(1, rates=(abs(lat)+12)*Hz)
        Synapses(P, G, on_pre='v += 5*mV').connect()
        mon = SpikeMonitor(G); run(40*ms)

        return {"sat": sat.name, "lat": lat, "spikes": int(mon.count[0])}

# --- 2. THE EXECUTION LOOP ---

hive = GeminiHive()
out = widgets.Output()
btn = widgets.ToggleButton(description="INITIATE GEMINI HIVE", button_style='primary')



async def core_loop():
    while btn.value:
        with out:
            clear_output(wait=True)
            state = hive.get_neuromorphic_state()
            print(f"📡 [Tracking] {state['sat']} | 🧠 Neural Activity: {state['spikes']} Spikes")

            # Equivocation Phase: All Gemini models deliberate at once
            context = {**state, "goal": "Equate Quantum Phase Shift"}
            tasks = [node.deliberate(context) for node in hive.nodes]
            results = await asyncio.gather(*tasks)

            # Consensus Integration
            valid = [r for r in results if r]
            if valid:
                # Weighted average using node-reported confidence
                final_theta = np.average([r['theta'] for r in valid], weights=[r['confidence'] for r in valid])
                print(f"⚖️ [Consensus] Equated {len(valid)} Gemini Nodes | Theta: {final_theta:.4f}")

                # Physical Layer: Quantum Gate Execution
                q = cirq.LineQubit(0)
                circuit = cirq.Circuit(cirq.ry(final_theta)(q), cirq.measure(q, key='m'))
                hist = qsimcirq.QSimSimulator().run(circuit, repetitions=100).histogram(key='m')
                print(f"⚛️ [Quantum] State Map: {hist}")

            await asyncio.sleep(2)

btn.observe(lambda c: asyncio.create_task(core_loop()) if c['new'] else None, 'value')
display(widgets.VBox([widgets.HTML("<h2>♊ Gemini Communicator Hive</h2>"), btn, out]))

In [ ]:
import asyncio, json, re, time
import google.generativeai as genai
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. THE GEMINI NODE CONTROLLER ---

class GeminiHiveNode:
    """Equates all Gemini variants as parallel peer nodes."""
    def __init__(self, model_id, api_key):
        genai.configure(api_key=api_key)
        self.model_id = model_id
        self.model = genai.GenerativeModel(model_id)

    async def deliberate(self, context):
        """Asynchronous node reasoning."""
        prompt = f"Satellite: {context['sat']}. Spikes: {context['spikes']}. Goal: {context['goal']}. Return JSON ONLY: {{'theta': float, 'confidence': float}}"

        loop = asyncio.get_event_loop()
        try:
            # Parallelize synchronous API call
            response = await loop.run_in_executor(None, lambda: self.model.generate_content(prompt))
            data = json.loads(re.search(r"\{.*\}", response.text, re.DOTALL).group())
            return {**data, "node": self.model_id}
        except:
            return None

class GeminiHive:
    def __init__(self):
        self.api_key = userdata.get('GEMINI_API_KEY')
        # Dynamic Equivocation: Find all supporting Gemini models
        available_ids = [m.name for m in genai.list_models()
                         if 'generateContent' in m.supported_generation_methods]

        self.nodes = [GeminiHiveNode(mid, self.api_key) for mid in available_ids]
        self.ts = load.timescale()

    def get_neuromorphic_state(self):
        """Phase 2: Sensory Spike Core"""
        start_scope()
        url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
        sat = load.tle_file(url)[0]
        lat = sat.at(self.ts.now()).subpoint().latitude.degrees

        # Unit-Consistent SNN
        G = NeuronGroup(1, 'dv/dt = (0*volt - v) / 10*ms : volt', threshold='v > 15*mV', reset='v = 0*volt', method='exact')
        P = PoissonGroup(1, rates=(abs(lat)+12)*Hz)
        Synapses(P, G, on_pre='v += 5*mV').connect()
        mon = SpikeMonitor(G); run(40*ms)

        return {"sat": sat.name, "lat": lat, "spikes": int(mon.count[0])}

# --- 2. THE EXECUTION LOOP ---

hive = GeminiHive()
out = widgets.Output()
btn = widgets.ToggleButton(description="INITIATE GEMINI HIVE", button_style='primary')



async def core_loop():
    while btn.value:
        with out:
            clear_output(wait=True)
            state = hive.get_neuromorphic_state()
            print(f"📡 [Tracking] {state['sat']} | 🧠 Neural Activity: {state['spikes']} Spikes")

            # Equivocation Phase: All Gemini models deliberate at once
            context = {**state, "goal": "Equate Quantum Phase Shift"}
            tasks = [node.deliberate(context) for node in hive.nodes]
            results = await asyncio.gather(*tasks)

            # Consensus Integration
            valid = [r for r in results if r]
            if valid:
                # Weighted average using node-reported confidence
                final_theta = np.average([r['theta'] for r in valid], weights=[r['confidence'] for r in valid])
                print(f"⚖️ [Consensus] Equated {len(valid)} Gemini Nodes | Theta: {final_theta:.4f}")

                # Physical Layer: Quantum Gate Execution
                q = cirq.LineQubit(0)
                circuit = cirq.Circuit(cirq.ry(final_theta)(q), cirq.measure(q, key='m'))
                hist = qsimcirq.QSimSimulator().run(circuit, repetitions=100).histogram(key='m')
                print(f"⚛️ [Quantum] State Map: {hist}")

            await asyncio.sleep(2)

btn.observe(lambda c: asyncio.create_task(core_loop()) if c['new'] else None, 'value')
display(widgets.VBox([widgets.HTML("<h2>♊ Gemini Communicator Hive</h2>"), btn, out]))

In [ ]:
DOnt't do it







No execution

In [ ]:
# 2. Install in a specific order to ensure dependency resolution
# We pin Sympy to a stable version compatible with Cirq 2026
!pip install sympy
!pip install cirq qsimcirq brian2 skyfield google-generativeai

In [ ]:
import sys

# Pre-flight check
try:
    import sympy.printing
    print("✅ SymPy environment healthy.")
except AttributeError:
    print("❌ SymPy is still corrupted. Please go to Runtime > Restart Session.")

# Consolidated Hive Imports
import json, re, asyncio, time
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
import asyncio, json, re, time
import google.generativeai as genai
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. THE GEMINI NODE CONTROLLER ---

class GeminiHiveNode:
    """Equates all Gemini variants as parallel peer nodes."""
    def __init__(self, model_id, api_key):
        genai.configure(api_key=api_key)
        self.model_id = model_id
        self.model = genai.GenerativeModel(model_id)

    async def deliberate(self, context):
        """Asynchronous node reasoning."""
        prompt = f"Satellite: {context['sat']}. Spikes: {context['spikes']}. Goal: {context['goal']}. Return JSON ONLY: {{'theta': float, 'confidence': float}}"

        loop = asyncio.get_event_loop()
        try:
            # Parallelize synchronous API call
            response = await loop.run_in_executor(None, lambda: self.model.generate_content(prompt))
            data = json.loads(re.search(r"\{.*\}", response.text, re.DOTALL).group())
            return {**data, "node": self.model_id}
        except:
            return None

class GeminiHive:
    def __init__(self):
        self.api_key = userdata.get('GEMINI_API_KEY')
        # Dynamic Equivocation: Find all supporting Gemini models
        available_ids = [m.name for m in genai.list_models()
                         if 'generateContent' in m.supported_generation_methods]

        self.nodes = [GeminiHiveNode(mid, self.api_key) for mid in available_ids]
        self.ts = load.timescale()

    def get_neuromorphic_state(self):
        """Phase 2: Sensory Spike Core"""
        start_scope()
        url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
        sat = load.tle_file(url)[0]
        lat = sat.at(self.ts.now()).subpoint().latitude.degrees

        # Unit-Consistent SNN
        G = NeuronGroup(1, 'dv/dt = (0*volt - v) / 10*ms : volt', threshold='v > 15*mV', reset='v = 0*volt', method='exact')
        P = PoissonGroup(1, rates=(abs(lat)+12)*Hz)
        Synapses(P, G, on_pre='v += 5*mV').connect()
        mon = SpikeMonitor(G); run(40*ms)

        return {"sat": sat.name, "lat": lat, "spikes": int(mon.count[0])}

# --- 2. THE EXECUTION LOOP ---

hive = GeminiHive()
out = widgets.Output()
btn = widgets.ToggleButton(description="INITIATE GEMINI HIVE", button_style='primary')



async def core_loop():
    while btn.value:
        with out:
            clear_output(wait=True)
            state = hive.get_neuromorphic_state()
            print(f"📡 [Tracking] {state['sat']} | 🧠 Neural Activity: {state['spikes']} Spikes")

            # Equivocation Phase: All Gemini models deliberate at once
            context = {**state, "goal": "Equate Quantum Phase Shift"}
            tasks = [node.deliberate(context) for node in hive.nodes]
            results = await asyncio.gather(*tasks)

            # Consensus Integration
            valid = [r for r in results if r]
            if valid:
                # Weighted average using node-reported confidence
                final_theta = np.average([r['theta'] for r in valid], weights=[r['confidence'] for r in valid])
                print(f"⚖️ [Consensus] Equated {len(valid)} Gemini Nodes | Theta: {final_theta:.4f}")

                # Physical Layer: Quantum Gate Execution
                q = cirq.LineQubit(0)
                circuit = cirq.Circuit(cirq.ry(final_theta)(q), cirq.measure(q, key='m'))
                hist = qsimcirq.QSimSimulator().run(circuit, repetitions=100).histogram(key='m')
                print(f"⚛️ [Quantum] State Map: {hist}")

            await asyncio.sleep(2)

btn.observe(lambda c: asyncio.create_task(core_loop()) if c['new'] else None, 'value')
display(widgets.VBox([widgets.HTML("<h2>♊ Gemini Communicator Hive</h2>"), btn, out]))

In [ ]:
import json, re, asyncio, time
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

class GeminiHiveNode:
    """A peer communicator node that uses a specific Gemini model."""
    def __init__(self, model_id, api_key):
        genai.configure(api_key=api_key)
        self.model_id = model_id
        self.model = genai.GenerativeModel(model_id)

    async def deliberate(self, context):
        """Asynchronous node deliberation."""
        prompt = f"Data: {context}. Output JSON ONLY: {{'theta': float (0-3.14), 'confidence': float}}"
        loop = asyncio.get_event_loop()
        try:
            # Running synchronous API in thread pool to prevent blocking
            response = await loop.run_in_executor(None, lambda: self.model.generate_content(prompt))
            return json.loads(re.search(r"\{.*\}", response.text, re.DOTALL).group())
        except:
            return None

class GeminiHive:
    def __init__(self):
        self.api_key = userdata.get('GEMINI_API_KEY')
        # Equivocation: Auto-discover all models for this key
        available = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
        self.nodes = [GeminiHiveNode(name, self.api_key) for name in available]
        self.ts = load.timescale()

    def get_neuromorphic_state(self):
        start_scope()
        url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
        sat = load.tle_file(url)[0]
        lat = sat.at(self.ts.now()).subpoint().latitude.degrees
        # Unit-Corrected equations to prevent DimensionMismatchError
        G = NeuronGroup(1, 'dv/dt = (0*volt - v) / 10*ms : volt', threshold='v > 15*mV', reset='v = 0*volt', method='exact')
        P = PoissonGroup(1, rates=(abs(lat)+10)*Hz)
        Synapses(P, G, on_pre='v += 5*mV').connect()
        mon = SpikeMonitor(G); run(30*ms)
        return {"sat": sat.name, "lat": lat, "spikes": int(mon.count[0])}

# --- EXECUTION INTERFACE ---

hive = GeminiHive()
out = widgets.Output()
btn = widgets.ToggleButton(description="INITIATE GEMINI HIVE", button_style='primary')

async def execution_loop():
    while btn.value:
        with out:
            clear_output(wait=True)
            state = hive.get_neuromorphic_state()
            print(f"📡 [Global] Tracking {state['sat']} | {state['spikes']} Neural Spikes")

            # Equivocation: Gather all node outputs simultaneously
            tasks = [node.deliberate(state) for node in hive.nodes]
            results = [r for r in await asyncio.gather(*tasks) if r]

            if results:
                # Weighted average based on node confidence
                final_theta = np.average([r['theta'] for r in results], weights=[r['confidence'] for r in results])
                print(f"⚖️ [Consensus] {len(results)} Nodes Equated | Result: {final_theta:.4f}")

                # Quantum physical layer execution
                q = cirq.LineQubit(0)
                circuit = cirq.Circuit(cirq.ry(final_theta)(q), cirq.measure(q, key='m'))
                hist = qsimcirq.QSimSimulator().run(circuit, repetitions=100).histogram(key='m')
                print(f"⚛️ [Quantum Output] {hist}")

            await asyncio.sleep(1)

btn.observe(lambda c: asyncio.create_task(execution_loop()) if c['new'] else None, 'value')
display(widgets.VBox([widgets.HTML("<h2>♊ Gemini Hive Node Engine</h2>"), btn, out]))

In [ ]:
import json, re, asyncio, time
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

class GeminiHiveNode:
    def __init__(self, model_id, api_key):
        # Local configuration for each node to ensure separation
        genai.configure(api_key=api_key)
        self.model_id = model_id
        self.model = genai.GenerativeModel(model_id)

    async def deliberate(self, context):
        prompt = f"Data: {context}. Return JSON ONLY: {{'theta': float (0-3.14), 'confidence': float}}"
        loop = asyncio.get_event_loop()
        try:
            # We use the thread pool to keep the UI responsive while waiting for API
            response = await loop.run_in_executor(None, lambda: self.model.generate_content(prompt))
            return json.loads(re.search(r"\{.*\}", response.text, re.DOTALL).group())
        except:
            return None

class GeminiHive:
    def __init__(self):
        try:
            # Step 1: Securely retrieve the key
            self.api_key = userdata.get('GEMINI_API_KEY')
            genai.configure(api_key=self.api_key)

            # Step 2: Equivocation - gather all node identities
            # This is where your error was occurring because of missing config
            available = [m.name for m in genai.list_models()
                         if 'generateContent' in m.supported_generation_methods]

            self.nodes = [GeminiHiveNode(name, self.api_key) for name in available]
            print(f"✅ Hive Synchronized: {len(self.nodes)} nodes online.")
        except Exception as e:
            print(f"❌ Initialization Failed: {e}")
            self.nodes = []

        self.ts = load.timescale()

    def get_neural_telemetry(self):
        start_scope()
        url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
        sat = load.tle_file(url)[0]
        lat = sat.at(self.ts.now()).subpoint().latitude.degrees

        # Neural Spike Layer (Unit Consistent)
        G = NeuronGroup(1, 'dv/dt = (0*volt - v) / 10*ms : volt', threshold='v > 15*mV', reset='v = 0*volt', method='exact')
        P = PoissonGroup(1, rates=(abs(lat)+12)*Hz)
        Synapses(P, G, on_pre='v += 5*mV').connect()
        mon = SpikeMonitor(G); run(30*ms)
        return {"sat": sat.name, "lat": lat, "spikes": int(mon.count[0])}

# --- INTERFACE ---
hive = GeminiHive()
out = widgets.Output()
btn = widgets.ToggleButton(description="INITIATE HIVE", button_style='info')



async def loop_execution():
    while btn.value:
        with out:
            clear_output(wait=True)
            state = hive.get_neural_telemetry()
            print(f"📡 Tracking {state['sat']} | Spikes: {state['spikes']}")

            # Parallel Deliberation: All nodes deliberate at once
            tasks = [node.deliberate(state) for node in hive.nodes]
            results = [r for r in await asyncio.gather(*tasks) if r]

            if results:
                # Weighted Integration
                theta = np.average([r['theta'] for r in results], weights=[r['confidence'] for r in results])
                print(f"⚖️ Hive Consensus: {theta:.4f}")

                # Quantum Phase Shift
                q = cirq.LineQubit(0)
                circuit = cirq.Circuit(cirq.ry(theta)(q), cirq.measure(q, key='m'))
                hist = qsimcirq.QSimSimulator().run(circuit, repetitions=100).histogram(key='m')
                print(f"⚛️ Quantum State: {hist}")

            await asyncio.sleep(2)

btn.observe(lambda c: asyncio.create_task(loop_execution()) if c['new'] else None, 'value')
display(widgets.VBox([widgets.HTML("<h2>🌌 Decentralized Gemini Hive</h2>"), btn, out]))

In [ ]:
import json, re, asyncio, time
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- PHASE 1: THE DYNAMIC HIVE NODE ---

class UniversalGeminiNode:
    """A peer node representing a single model in the Gemini family."""
    def __init__(self, model_id, api_key):
        self.model_id = model_id
        # Individual configuration per node for thread safety
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model_id)

    async def deliberate(self, context, semaphore):
        """Asynchronous deliberation with rate-limit protection."""
        async with semaphore:
            prompt = f"""
HIVE CONTEXT: Satellite {context['sat']} | Neural Spikes: {context['spikes']}
OBJECTIVE: Calculate optimal Quantum Phase (theta).
CONSTRAINT: Output strictly valid JSON.
JSON FORMAT: {{"theta": float, "confidence": float, "reasoning": "string"}}
            """
            loop = asyncio.get_event_loop()
            try:
                # Use thread-pool executor for the synchronous SDK call
                response = await loop.run_in_executor(None, lambda: self.model.generate_content(prompt))
                match = re.search(r"\{.*\}", response.text, re.DOTALL)
                data = json.loads(match.group())
                return {**data, "node": self.model_id}
            except Exception:
                return None

class GeminiUniversalHive:
    def __init__(self):
        self.api_key = userdata.get('GEMINI_API_KEY')
        genai.configure(api_key=self.api_key)

        # Discovery: Find ALL models supporting generation
        # This includes 3 Pro, 3 Flash, 2.5 Pro, etc.
        models = genai.list_models()
        self.node_ids = [m.name for m in models if 'generateContent' in m.supported_generation_methods]

        self.nodes = [UniversalGeminiNode(mid, self.api_key) for mid in self.node_ids]
        # Semaphore limits concurrent API hits to prevent 429 Rate Limit errors
        self.semaphore = asyncio.Semaphore(len(self.nodes))
        self.ts = load.timescale()

    def get_neuromorphic_state(self):
        start_scope()
        sat = load.tle_file('https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle')[0]
        lat = sat.at(self.ts.now()).subpoint().latitude.degrees

        # Spiking Neural Core
        G = NeuronGroup(1, 'dv/dt = (0*volt - v) / 10*ms : volt', threshold='v > 15*mV', reset='v = 0*volt', method='exact')
        P = PoissonGroup(1, rates=(abs(lat)+15)*Hz)
        Synapses(P, G, on_pre='v += 6*mV').connect()
        mon = SpikeMonitor(G); run(35*ms)
        return {"sat": sat.name, "lat": lat, "spikes": int(mon.count[0])}

# --- PHASE 2: HIVE EXECUTION & QUANTUM INTEGRATION ---

hive = GeminiUniversalHive()
out = widgets.Output()
btn = widgets.ToggleButton(description="ACTIVATE ALL NODES", button_style='success')

async def execution_loop():
    while btn.value:
        with out:
            clear_output(wait=True)
            state = hive.get_neuromorphic_state()
            print(f"📡 [Tracking] {state['sat']} | 🧠 Neural Pulse: {state['spikes']} Hz")
            print(f"🐝 [Hive] Equating {len(hive.nodes)} Gemini models...")

            # EQUIVOCATION: Fire all models at once
            tasks = [node.deliberate(state, hive.semaphore) for node in hive.nodes]
            results = [r for r in await asyncio.gather(*tasks) if r]

            if results:
                # CONSENSUS: Use confidence-weighted average for theta
                weights = [r['confidence'] for r in results]
                thetas = [r['theta'] for r in results]
                final_theta = np.average(thetas, weights=weights)

                print(f"⚖️ [Consensus] Result: {final_theta:.4f} | Nodes Active: {len(results)}")

                # PHYSICAL LAYER: Quantum Circuit
                q = cirq.LineQubit(0)
                circuit = cirq.Circuit(cirq.ry(final_theta)(q), cirq.measure(q, key='m'))
                hist = qsimcirq.QSimSimulator().run(circuit, repetitions=100).histogram(key='m')
                print(f"⚛️ [Quantum] Probabilities: {hist}")

            await asyncio.sleep(2)

btn.observe(lambda c: asyncio.create_task(execution_loop()) if c['new'] else None, 'value')
display(widgets.VBox([widgets.HTML("<h2>🌌 Universal Gemini Communicator Hive</h2>"), btn, out]))

In [ ]:
!pip install google-generativeai

In [ ]:
import concurrent.futures
import json, re, cirq, qsimcirq, numpy as np
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# Configure Gemini API globally once
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

class HiveEngine:
    def __init__(self):
        # Store API key
        self.api_key = userdata.get('GEMINI_API_KEY')
        # Ensure genai is configured before listing models
        genai.configure(api_key=self.api_key)
        self.available_models = [m.name for m in genai.list_models()
                                if 'generateContent' in m.supported_generation_methods]
        self.ts = load.timescale()

    def fetch_satellite(self):
        url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
        sat = load.tle_file(url)[0]
        sub = sat.at(self.ts.now()).subpoint()
        return {"name": sat.name, "lat": sub.latitude.degrees}

    def process_spikes(self, lat):
        start_scope()
        tau, v_rest, v_thresh = 10*ms, 0*volt, 0.015*volt
        G = NeuronGroup(1, 'dv/dt = (v_rest - v) / tau : volt', threshold='v > v_thresh', reset='v = v_rest', method='exact')
        input_p = PoissonGroup(1, rates=(abs(lat) + 15)*Hz)
        Synapses(input_p, G, on_pre='v += 0.005*volt').connect()
        mon = SpikeMonitor(G)
        run(50*ms)
        return int(mon.count[0])

    def call_model(self, model_name, prompt):
        """Single thread worker for a specific model."""
        try:
            # API key is already configured globally, so it's not strictly necessary
            # to pass it here, but it's a safe explicit approach if global config is flaky.
            model = genai.GenerativeModel(model_name, api_key=self.api_key)
            res = model.generate_content(prompt)
            data = json.loads(re.search(r"\{.*\}", res.text, re.DOTALL).group())
            return {"model": model_name, "theta": data['theta'], "status": "OK"}
        except Exception as e:
            return {"model": model_name, "status": f"Error: {str(e)}"}

# --- INTERFACE & EXECUTION ---

engine = HiveEngine()
output_log = widgets.Output()
btn = widgets.Button(description="ACTIVATE HIVE MIND", button_style='danger', layout={'width':'100%'})

In [ ]:
# --- import asyncio
import concurrent.futures
import json, re, time
import numpy as np
import cirq, qsimcirq
from brian2 import *
from skyfield.api import load
import google.generativeai as genai
from google.colab import userdata
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- PHASE 1: HIVE NODE ARCHITECTURE ---

class HiveNode:
    """A decentralized node that processes data independently."""
    def __init__(self, model_name, api_key):
        genai.configure(api_key=api_key)
        self.name = model_name
        self.model = genai.GenerativeModel(model_name)

    async def deliberate(self, context):
        """Asynchronous deliberation with the Hive context."""
        prompt = f"""
        HIVE NODE IDENTIFIER: {self.name}
        TELEMETRY: {context['sat_name']} at {context['lat']:.2f}
        NEURAL STATE: {context['spikes']} spikes detected.
        MISSION GOAL: {context['goal']}

        Deliberate on the optimal Quantum Rotation (theta).
        Return ONLY valid JSON: {{"node_theta": float, "confidence": float}}
        """
        # Run synchronous API call in an executor to avoid blocking the loop
        loop = asyncio.get_event_loop()
        try:
            response = await loop.run_in_executor(None, lambda: self.model.generate_content(prompt))
            match = re.search(r"\{.*\}", response.text, re.DOTALL)
            return json.loads(match.group()) if match else None
        except:
            return None

class HiveCommunicator:
    """The Hive Controller that equates and integrates all nodes."""
    def __init__(self):
        self.api_key = userdata.get('GEMINI_API_KEY')
        genai.configure(api_key=self.api_key)
        # Auto-equate: Find all supported models
        available = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
        self.nodes = [HiveNode(name, self.api_key) for name in available]
        self.ts = load.timescale()
        self.active = False

    def get_telemetry(self):
        url = 'https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle'
        sat = load.tle_file(url)[0]
        pos = sat.at(self.ts.now()).subpoint()
        return {"name": sat.name, "lat": pos.latitude.degrees}

    def get_neuromorphic_state(self, lat):
        start_scope()
        freq = (abs(lat) + 10) * Hz
        G = NeuronGroup(1, 'dv/dt = (0*volt - v) / 10*ms : volt', threshold='v > 15*mV', reset='v = 0*volt', method='exact')
        P = PoissonGroup(1, rates=freq)
        Synapses(P, G, on_pre='v += 5*volt/1000').connect() # Using base volts
        mon = SpikeMonitor(G)
        run(30*ms)
        return int(mon.count[0])

# --- PHASE 2: EXECUTION & INTEGRATION ---

hive = HiveCommunicator()
out = widgets.Output()
run_btn = widgets.ToggleButton(value=False, description="ACTIVATE HIVE NODES", button_style='info')



async def core_execution_loop():
    while run_btn.value:
        with out:
            clear_output(wait=True)
            # 1. Gather environmental state
            tel = hive.get_telemetry()
            spikes = hive.get_neuromorphic_state(tel['lat'])

            context = {
                "sat_name": tel['name'],
                "lat": tel['lat'],
                "spikes": spikes,
                "goal": "Maintain Quantum Synchronicity"
            }

            print(f"📡 [Global Telemetry] Tracking {tel['name']} | Spikes: {spikes}")
            print(f"🐝 [Hive Status] {len(hive.nodes)} nodes active. Synchronizing...")

            # 2. Equivocation: All nodes deliberate in parallel
            tasks = [node.deliberate(context) for node in hive.nodes]
            results = await asyncio.gather(*tasks)

            # 3. Integration: Weighted Consensus
            valid_results = [r for r in results if r is not None]
            if valid_results:
                # Calculate theta weighted by node confidence
                weights = np.array([r['confidence'] for r in valid_results])
                thetas = np.array([r['node_theta'] for r in valid_results])
                consensus_theta = np.average(thetas, weights=weights/weights.sum())

                print(f"⚖️ [Consensus] Equated Theta: {consensus_theta:.4f}")

                # 4. Physical Layer Execution
                q_res = execute_quantum_gate(consensus_theta)
                print(f"⚛️ [Quantum] State Map: {q_res}")

            await asyncio.sleep(1) # Frequency of hive updates

def execute_quantum_gate(theta):
    q = cirq.LineQubit(0)
    circuit = cirq.Circuit(cirq.ry(theta)(q), cirq.measure(q, key='m'))
    return qsimcirq.QSimSimulator().run(circuit, repetitions=100).histogram(key='m')

def on_toggle(change):
    if change['new']:
        asyncio.create_task(core_execution_loop())

run_btn.observe(on_toggle, 'value')
display(widgets.VBox([widgets.HTML("<h2>🌌 Decentralized Hive Node Controller</h2>"), run_btn, out]))